# 归档材料

保留此材料用于局部机制学习；先修、结论与下游连接需要结合归档索引审查。

[归档索引](../README.md) · [当前学习入口](../../../course/first_loop/README.md)

# 01 · Sensors & Geometry：传感器怎样进入同一个世界？

`camera/LiDAR/radar/GNSS/IMU` 的观测不天然对齐。自动驾驶模型输入之前，需要明确 **frame、SE(3)、intrinsic/extrinsic、timestamp、ego-motion**。本章把旧版 `00B` 和 `01` 合并，并让输出成为下一章 BEV 的输入。

约定：ego frame 使用 `x forward, y left, z up`；相机 frame 使用 `z forward, x right, y down`。真实项目必须以数据集/车辆平台文档为准。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

scene = build_urban_cut_in_scene(seed=7, timestamp_s=0.0)
np.set_printoptions(precision=3, suppress=True)

def make_T(yaw_deg=0.0, translation=(0.0, 0.0, 0.0)):
    yaw = np.deg2rad(yaw_deg)
    c, s = np.cos(yaw), np.sin(yaw)
    T = np.eye(4)
    T[:3, :3] = [[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]]
    T[:3, 3] = np.asarray(translation, dtype=float)
    return T

def transform(T, points_xyz):
    points_xyz = np.asarray(points_xyz)
    homogeneous = np.c_[points_xyz, np.ones(len(points_xyz))]
    return (T @ homogeneous.T).T[:, :3]

def project(K, points_camera):
    z = points_camera[:, 2]
    valid = z > 1e-6
    uv = np.full((len(points_camera), 2), np.nan)
    uv[valid] = (K @ points_camera[valid].T).T[:, :2] / z[valid, None]
    return uv, valid

R_ce = np.array([[0.0, -1.0, 0.0], [0.0, 0.0, -1.0], [1.0, 0.0, 0.0]])
camera_origin_in_ego = np.array([1.3, 0.0, 1.4])
T_c_from_e = np.eye(4)
T_c_from_e[:3, :3] = R_ce
T_c_from_e[:3, 3] = -R_ce @ camera_origin_in_ego
K = np.array([[720.0, 0.0, 640.0], [0.0, 720.0, 360.0], [0.0, 0.0, 1.0]])

camera_points = transform(T_c_from_e, scene.lidar_points)
uv, valid = project(K, camera_points)
print("valid projected points:", int(valid.sum()), "of", len(valid))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(scene.lidar_points[:, 0], scene.lidar_points[:, 1], s=4, alpha=0.35)
axes[0].set_aspect("equal")
axes[0].set(xlabel="ego x / m", ylabel="ego y / m", title="LiDAR in ego frame")
axes[1].scatter(uv[valid, 0], uv[valid, 1], s=4, alpha=0.35)
axes[1].invert_yaxis()
axes[1].set(xlim=(0, 1280), ylim=(720, 0), xlabel="u / px", ylabel="v / px", title="Projected observation")
plt.tight_layout()
plt.show()

## 1. Calibration and time are model errors, not just plumbing

一个小的 yaw/translation 误差会变成像素偏移，随后又变成 BEV cell 错位。时间错位还会把“同一个 actor 的不同位置”当成空间融合冲突。先预测曲线，再调整下面的扰动。

In [ ]:
yaw_errors = np.linspace(-2.0, 2.0, 41)
pixel_shift = []
uv_ref, ref_valid = project(K, transform(T_c_from_e, scene.lidar_points))
for error in yaw_errors:
    perturbed = make_T(error) @ T_c_from_e
    uv_error, error_valid = project(K, transform(perturbed, scene.lidar_points))
    both = ref_valid & error_valid
    pixel_shift.append(np.nanmean(np.linalg.norm(uv_error[both] - uv_ref[both], axis=1)))
plt.plot(yaw_errors, pixel_shift, marker=".")
plt.xlabel("injected extrinsic yaw error / deg")
plt.ylabel("mean pixel displacement / px")
plt.title("Calibration error becomes feature/fusion error")
plt.show()

for lag in [0.0, 0.05, 0.15, 0.30]:
    lagged = build_urban_cut_in_scene(seed=7, timestamp_s=0.0, sensor_lag_s=lag)
    dx = np.mean(lagged.lidar_points[:, 0]) - np.mean(lagged.camera_points[:, 0])
    print(f"camera lag={lag:.2f}s -> mean x discrepancy={dx:.3f}m")

## 2. Real-data checkpoint: nuScenes mini

Toy geometry is useful only if it transfers to a real schema. After downloading **nuScenes mini** under its own license, set `NUSCENES_ROOT` and run the following adapter. It uses the official devkit to fetch one `LIDAR_TOP` sample and one `CAM_FRONT` calibrated sensor, then asks you to compare the projection with the toy convention. The cell is intentionally skipped when the optional package/data are absent.

In [ ]:
import os

def nuScenes_projection_checkpoint():
    root = os.environ.get("NUSCENES_ROOT")
    if not root:
        print("Set NUSCENES_ROOT to run the real-data checkpoint; toy experiment remains reproducible.")
        return
    try:
        from nuscenes.nuscenes import NuScenes
        from nuscenes.utils.data_classes import LidarPointCloud
        from pyquaternion import Quaternion
        from PIL import Image
    except ImportError as exc:
        print("Install requirements-real-data.txt first:", exc)
        return

    nusc = NuScenes(version="v1.0-mini", dataroot=root, verbose=False)
    sample = nusc.sample[0]
    lidar_sd = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
    camera_sd = nusc.get("sample_data", sample["data"]["CAM_FRONT"])
    lidar_cs = nusc.get("calibrated_sensor", lidar_sd["calibrated_sensor_token"])
    camera_cs = nusc.get("calibrated_sensor", camera_sd["calibrated_sensor_token"])
    lidar_pose = nusc.get("ego_pose", lidar_sd["ego_pose_token"])
    camera_pose = nusc.get("ego_pose", camera_sd["ego_pose_token"])

    # Official nuScenes frame chain: lidar sensor → lidar ego → global
    # → camera ego → camera sensor.  The order is part of the result.
    cloud = LidarPointCloud.from_file(str(Path(root) / lidar_sd["filename"]))
    cloud.rotate(Quaternion(lidar_cs["rotation"]).rotation_matrix)
    cloud.translate(np.asarray(lidar_cs["translation"]))
    cloud.rotate(Quaternion(lidar_pose["rotation"]).rotation_matrix)
    cloud.translate(np.asarray(lidar_pose["translation"]))
    cloud.translate(-np.asarray(camera_pose["translation"]))
    cloud.rotate(Quaternion(camera_pose["rotation"]).inverse.rotation_matrix)
    cloud.translate(-np.asarray(camera_cs["translation"]))
    cloud.rotate(Quaternion(camera_cs["rotation"]).inverse.rotation_matrix)

    camera_points = cloud.points[:3].T
    uv_real, valid_real = project(np.asarray(camera_cs["camera_intrinsic"]), camera_points)
    image_width, image_height = Image.open(Path(root) / camera_sd["filename"]).size
    in_image = valid_real & (uv_real[:, 0] >= 0) & (uv_real[:, 0] < image_width) & (uv_real[:, 1] >= 0) & (uv_real[:, 1] < image_height)
    result = {"sample": sample["token"], "points": int(len(camera_points)), "projected_in_image": int(in_image.sum()),
              "image_size": [image_width, image_height], "frame_chain": "lidar→ego→global→camera_ego→camera"}
    print(result)
    return camera_points, uv_real, in_image

nuScenes_projection_checkpoint()

### 完成标准

解释 `T_camera←ego` 的旋转和平移；画出一条“标定/时间错误 → BEV 错位 → 模型后果”的链；并完成一次 real-data checkpoint 或记录缺少的数据/依赖。下一章会消费同一个 `scene`，将传感器观测栅格化为 BEV。

In [ ]:
save_json_artifact("01_geometry.json", {
    "scene_id": scene.scene_id,
    "frame_convention": {"ego": "x forward, y left, z up", "camera": "z forward, x right, y down"},
    "projected_points": int(valid.sum()),
    "calibration_sensitivity_px_per_deg": float(np.polyfit(yaw_errors, pixel_shift, 1)[0]),
    "next": "02_bev_and_fusion.ipynb",
})
print("saved geometry artifact")